In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"]="6"
DEVICE = "cuda"
BATCH_SIZE = 32

import torch

In [ ]:
from open_vocab_mot.definitions import DUKEMTMC_VIDEO_REID_PATH, DUKEMTMC_VIDEO_REID_SIDECAR_PATH
from open_vocab_mot.data import DukePersonId, DukeCameraId, DukeFrameName, DukeSplit, DukeMTMCVideoDataset, collate_duke_mtmc_video_ds, DukeMTMCItemBatch

In [ ]:
print(DUKEMTMC_VIDEO_REID_PATH)
print(DUKEMTMC_VIDEO_REID_SIDECAR_PATH)

In [ ]:
duke_ds = DukeMTMCVideoDataset(DUKEMTMC_VIDEO_REID_PATH, main_split=DukeSplit.TRAIN, verbose=True)

In [ ]:
from torch.utils.data import DataLoader
duke_loader = DataLoader(duke_ds, batch_size=BATCH_SIZE, collate_fn=collate_duke_mtmc_video_ds)

In [ ]:
batch = next(iter(duke_loader))

In [ ]:
from torchvision.io import read_image, ImageReadMode
from pathlib import Path

def load_imgs_and_segs(frame_paths: list[Path], ds_root: Path, sidecar_root: Path):
    # Load the images and segmentations
    imgs = []
    segs = []
    for p in frame_paths:
        img = read_image(str(p), ImageReadMode.RGB)
        imgs.append((img.float() / 255.0).to(DEVICE))

        # Get the segmentation path
        relative_parent_path = p.parent.relative_to(ds_root)
        seg_path = sidecar_root / relative_parent_path / f"{p.stem}_major_mask.png"
        assert seg_path.exists(), f"Segmentation not found at {seg_path}"
        seg = read_image(str(seg_path), ImageReadMode.GRAY)
        segs.append(seg)

    return imgs, segs

In [ ]:
imgs, segs = load_imgs_and_segs(batch.frame_paths, DUKEMTMC_VIDEO_REID_PATH, DUKEMTMC_VIDEO_REID_SIDECAR_PATH)

In [ ]:
print(imgs[0].shape)
print(segs[0].shape)

In [ ]:
from aidan_lib.models.dino_lib_compiled import DINOv3CompiledHarness

In [ ]:
dino_harness = DINOv3CompiledHarness(
    checkpoint="facebook/dinov3-vits16-pretrain-lvd1689m",
    device="cuda",
    dtype=torch.bfloat16,
    max_side_len=1024,
    warmup_batch_size=BATCH_SIZE,
    warmup=True
)

In [ ]:
print(imgs[0].dtype, imgs[0].device)
print(segs[0].dtype, segs[0].device)

In [ ]:
dino_out = dino_harness.match_bool_segmentations_to_dino(imgs, segs)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA
import torch

def visualize_dino_segments(imgs, dino_out, img_idx=0, alpha_blend=0.5):
    # Collect all embeddings across the batch to fit PCA for consistent colors
    all_embeddings = []
    for out in dino_out:
        if len(out) > 0:
            # We convert to float32 first since numpy might not support bfloat16 directly
            all_embeddings.append(out[0].dino_embeddings.cpu().float().numpy())
    
    if len(all_embeddings) == 0:
        print("No segmentations found in the entire batch.")
        return
        
    all_embeddings = np.concatenate(all_embeddings, axis=0)
    
    # Run PCA to reduce to 3 dimensions
    pca = PCA(n_components=3)
    pca.fit(all_embeddings)
    
    # Prepare the original image for plotting
    img = imgs[img_idx].cpu().float().permute(1, 2, 0).numpy() # (H, W, 3)
    
    if len(dino_out[img_idx]) == 0:
        print("No segmentations found for this image.")
        plt.figure(figsize=(12, 12))
        plt.imshow(img)
        plt.axis('off')
        plt.show()
        return

    seg = dino_out[img_idx][0]
    embeddings = seg.dino_embeddings.cpu().float().numpy() # (N, dim)
    bboxes = seg.dino_bboxes.cpu().numpy() # (N, 4)
    
    # Transform the current image's embeddings using the PCA fitted on the batch
    reduced_embeddings = pca.transform(embeddings) # (N, 3)
    
    # Normalize to [0, 1] using global min/max for true color consistency across the batch
    all_reduced = pca.transform(all_embeddings)
    min_vals = all_reduced.min(axis=0)
    max_vals = all_reduced.max(axis=0)
    
    reduced_embeddings = (reduced_embeddings - min_vals) / (max_vals - min_vals + 1e-6)
    
    # Create an overlay image and an alpha mask
    overlay = np.zeros_like(img)
    alpha = np.zeros(img.shape[:2], dtype=np.float32)
    
    for i in range(len(bboxes)):
        px1, py1, px2, py2 = bboxes[i]
        
        # Ensure we don't go out of bounds
        px1 = max(0, min(px1, img.shape[1] - 1))
        py1 = max(0, min(py1, img.shape[0] - 1))
        px2 = max(0, min(px2, img.shape[1]))
        py2 = max(0, min(py2, img.shape[0]))
        
        color = reduced_embeddings[i]
        
        # Fill the patch area with the PCA color
        overlay[py1:py2, px1:px2] = color
        alpha[py1:py2, px1:px2] = alpha_blend # Apply opacity to patched areas
    
    # Blend the original image and the overlay
    # Where alpha is 0 (no overlap), it will just be the original image
    alpha = np.expand_dims(alpha, axis=-1)
    blended = img * (1 - alpha) + overlay * alpha
    
    # Clip to be safe for matplotlib
    blended = np.clip(blended, 0, 1)
    
    # Plot the result
    plt.figure(figsize=(12, 12))
    plt.imshow(blended)
    plt.title(f"Image {img_idx} with DINO PCA Segmentations")
    plt.axis('off')
    plt.show()


In [ ]:
# Call the function to visualize the first image in the batch
visualize_dino_segments(imgs, dino_out, img_idx=15)